In [1]:
import torch
from torch_openreml.covariance import Sum, ScalarMatrix

x = Sum(ScalarMatrix(2), ScalarMatrix(2))
x

Sum(
  operands={
    'op_0': ScalarMatrix(shape=(2, 2), param_specs={'sigma^2': {'fixed': False, 'default': tensor([0.]), 'trans': TransformExpPow2()}}),
    'op_1': ScalarMatrix(shape=(2, 2), param_specs={'sigma^2': {'fixed': False, 'default': tensor([0.]), 'trans': TransformExpPow2()}})
  }
)

In [2]:
x = Sum(A = ScalarMatrix(2), B = ScalarMatrix(2))
x

Sum(
  operands={
    'A': ScalarMatrix(shape=(2, 2), param_specs={'sigma^2': {'fixed': False, 'default': tensor([0.]), 'trans': TransformExpPow2()}}),
    'B': ScalarMatrix(shape=(2, 2), param_specs={'sigma^2': {'fixed': False, 'default': tensor([0.]), 'trans': TransformExpPow2()}})
  }
)

In [3]:
x.param_names

['A/sigma^2', 'B/sigma^2']

In [4]:
x(torch.zeros(2))

tensor([[2., 0.],
        [0., 2.]])

In [5]:
x({"A/sigma^2": torch.zeros(1), "B/sigma^2": torch.zeros(1)})

tensor([[2., 0.],
        [0., 2.]])

In [6]:
import torch
from torch_openreml.covariance import Sum, ScalarMatrix

x = Sum(ScalarMatrix(2), ScalarMatrix(2))
free_params = torch.tensor([0.0, 0.5])

x.build_params(free_params)

tensor([1.0000, 2.7183])

In [7]:
x.build_params()

tensor([1., 1.])

In [8]:
x.build_params(free_params, trans=False)

tensor([0.0000, 0.5000])

In [9]:
x.build_params(free_params, out_format="dict")

{'op_0/sigma^2': tensor(1.), 'op_1/sigma^2': tensor(2.7183)}

In [10]:
import torch
from torch_openreml.covariance import Sum, ScalarMatrix

x = Sum(ScalarMatrix(2), ScalarMatrix(2))
v_groups = x.build_operands(torch.tensor([1.0, 2.0]))
print(v_groups[0])
print(v_groups[1])

tensor([[7.3891, 0.0000],
        [0.0000, 7.3891]])
tensor([[54.5982,  0.0000],
        [ 0.0000, 54.5982]])


In [11]:
import torch
from torch_openreml.covariance import BlockDiagonal, DiagonalMatrix, ScalarMatrix, Sum

op = Sum(inner=BlockDiagonal(DiagonalMatrix(2), ScalarMatrix(2)),
         extra=ScalarMatrix(4))
results, free_params_by_path = op.call_tree(torch.tensor([0.0, 0.5, 1.0, 0.5]))

In [12]:
results

{'/': tensor([[ 3.7183,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  5.4366,  0.0000,  0.0000],
         [ 0.0000,  0.0000, 10.1073,  0.0000],
         [ 0.0000,  0.0000,  0.0000, 10.1073]]),
 'inner': tensor([[1.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 2.7183, 0.0000, 0.0000],
         [0.0000, 0.0000, 7.3891, 0.0000],
         [0.0000, 0.0000, 0.0000, 7.3891]]),
 'inner/op_0': tensor([[1.0000, 0.0000],
         [0.0000, 2.7183]]),
 'inner/op_1': tensor([[7.3891, 0.0000],
         [0.0000, 7.3891]]),
 'extra': tensor([[2.7183, 0.0000, 0.0000, 0.0000],
         [0.0000, 2.7183, 0.0000, 0.0000],
         [0.0000, 0.0000, 2.7183, 0.0000],
         [0.0000, 0.0000, 0.0000, 2.7183]])}

In [13]:
results["inner/op_0"]

tensor([[1.0000, 0.0000],
        [0.0000, 2.7183]])

In [14]:
free_params_by_path["inner"]

tensor([0.0000, 0.5000, 1.0000])

In [15]:
import torch
from torch_openreml.covariance import BlockDiagonal, DiagonalMatrix, ScalarMatrix, Sum

op = Sum(inner=BlockDiagonal(DiagonalMatrix(2), ScalarMatrix(2)),
         extra=ScalarMatrix(4))
grads, free_params_by_path = op.grad_tree(torch.tensor([0.0, 0.5, 1.0, 0.5]))

In [16]:
grads["inner/op_0"]

(tensor([[[2.0000, 0.0000],
          [0.0000, 0.0000]],
 
         [[0.0000, 0.0000],
          [0.0000, 5.4366]]]),
 ['sigma^2_0', 'sigma^2_1'])

In [17]:
grads["inner"][1]

['op_0/sigma^2_0', 'op_0/sigma^2_1', 'op_1/sigma^2']

In [18]:
import torch
from torch_openreml.covariance import Sum, ScalarMatrix

x = Sum(ScalarMatrix(2), ScalarMatrix(2))
grad_groups, grad_name_groups = x.operands_grad(torch.tensor([1.0, 2.0]))
print(grad_groups[0])
print(grad_groups[1])
print(grad_name_groups[0])
print(grad_name_groups[1])

tensor([[[14.7781,  0.0000],
         [ 0.0000, 14.7781]]])
tensor([[[109.1963,   0.0000],
         [  0.0000, 109.1963]]])
['op_0/sigma^2']
['op_1/sigma^2']
